# EDA: regime detection (single-condition vs. 6-condition)

Exploration only, nothing here is load-bearing. Supports decisions made in `src/regimes.py`.

**Question 1:** does k=6 actually hold up in the raw operational-setting data for FD002/FD004, or was that just trusting the doc?

**Question 2:** `regimes.py` needs to auto-detect whether a dataset is single-condition (FD001/FD003, no-op) or 6-condition (FD002/FD004, k-means) using a std threshold on op1/op2/op3. What's a safe, well-justified threshold, and how much margin does it actually have?

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src.load import load_train

%matplotlib inline

## Question 1: is k=6 real?

Round FD002's op-settings and count how many distinct combinations actually show up, with how many rows each.

In [2]:
df2 = load_train("FD002")
rounded = df2[["op1", "op2", "op3"]].round(1)
print(rounded.value_counts())

op1   op2  op3  
42.0  0.8  100.0    13458
20.0  0.7  100.0     8122
0.0   0.0  100.0     8044
35.0  0.8  100.0     8037
25.0  0.6  60.0      8002
10.0  0.2  100.0     4138
      0.3  100.0     3958
Name: count, dtype: int64


**Finding:** 7 rounded groups show up, but two of them (`op1=10.0, op2=0.2` and `op1=10.0, op2=0.3`) are the same real regime split by floating-point noise landing right on a rounding boundary (true value near op2=0.25). The real count is 6, confirmed. k=6 is not a blind trust of the doc.

## Question 2: how safe is the single-condition/multi-condition threshold?

`_is_single_condition()` in `src/regimes.py` checks whether op1/op2/op3's standard deviation is below a threshold, for all three columns. Compare the worst case on each side: the noisiest single-condition column vs. the tightest multi-condition column.

In [3]:
import pandas as pd

rows = []
for name in ["FD001", "FD002", "FD003", "FD004"]:
    d = load_train(name)
    rows.append({"dataset": name, **d[["op1", "op2", "op3"]].std().to_dict()})

std_table = pd.DataFrame(rows).set_index("dataset")
print(std_table)

single_cols = std_table.loc[["FD001", "FD003"]]
multi_cols = std_table.loc[["FD002", "FD004"]]
print()
print("Single-condition worst (max) per column:")
print(single_cols.max())
print()
print("Multi-condition tightest (min) per column:")
print(multi_cols.min())

               op1       op2        op3
dataset                                
FD001     0.002187  0.000293   0.000000
FD002    14.747376  0.310016  14.237735
FD003     0.002194  0.000294   0.000000
FD004    14.780722  0.310703  14.251954

Single-condition worst (max) per column:
op1    0.002194
op2    0.000294
op3    0.000000
dtype: float64

Multi-condition tightest (min) per column:
op1    14.747376
op2     0.310016
op3    14.237735
dtype: float64


**Finding:** op1 and op3 have a huge gap either way (~0.002 vs. ~14+). op2 is the tight one: 0.0003 (single-condition) vs. 0.31 (multi-condition). At the originally-chosen threshold of 1.0, op2's multi-condition value (0.31) is actually *below* the threshold — op2 alone wouldn't correctly flag FD002/FD004 as multi-condition at that setting. The check still works today only because `_is_single_condition` requires *all three* columns to agree (`.all()`), and op1/op3 unambiguously do.

**Decision:** lower `SINGLE_CONDITION_STD_THRESHOLD` from 1.0 to 0.1. That sits comfortably between op2's two values (0.0003 and 0.31) as well as op1/op3's, so every column independently discriminates correctly — the classification no longer depends on the AND-across-columns logic to compensate for one close column.

## Question 3: what threshold correctly separates constant sensors from real signal?

`dataset-reference.md` lists FD001's constant sensors as s1, s5, s6, s10, s16, s18, s19 — but checking earlier (in `eda_features.ipynb`) found s6 is NOT flat in every unit: only 38 of 100 FD001 units have exactly zero per-unit variance for it, the other 62 have some small real movement. So "constant" isn't a clean binary in the raw data — a threshold needs to be chosen deliberately, not assumed.

This check uses the data as `features.py` will actually receive it (i.e. after `fit_regimes`, so FD002/FD004 sensors are already regime-normalized), and looks at the **worst-case (maximum) per-unit standard deviation** for each sensor — a sensor only counts as safely droppable if *no* unit shows meaningful movement in it.

In [4]:
from src.load import SENSOR_COLS
from src.regimes import fit_regimes

for name in ["FD001", "FD002", "FD003", "FD004"]:
    d = load_train(name)
    d_norm, _fitted = fit_regimes(d)

    max_per_unit_std = {}
    for s in SENSOR_COLS:
        per_unit_std = d_norm.groupby("unit")[s].std()
        max_per_unit_std[s] = per_unit_std.max()

    ranked = sorted(max_per_unit_std.items(), key=lambda x: x[1])
    print(f"=== {name}: max per-unit std, sorted (lowest = most constant) ===")
    for s, v in ranked[:10]:
        print(f"  {s}: {v:.6f}")
    print()

=== FD001: max per-unit std, sorted (lowest = most constant) ===
  s1: 0.000000
  s5: 0.000000
  s10: 0.000000
  s16: 0.000000
  s18: 0.000000
  s19: 0.000000
  s6: 0.002881
  s15: 0.042593
  s8: 0.089522
  s13: 0.092445



=== FD002: max per-unit std, sorted (lowest = most constant) ===
  s18: 0.000000
  s5: 0.000000
  s19: 0.000000
  s1: 0.000000
  s16: 0.556526
  s21: 1.148041
  s2: 1.148383
  s15: 1.154195
  s20: 1.154574
  s4: 1.163390

=== FD003: max per-unit std, sorted (lowest = most constant) ===
  s1: 0.000000
  s5: 0.000000
  s16: 0.000000
  s18: 0.000000
  s19: 0.000000
  s10: 0.005474
  s6: 0.036944
  s15: 0.056742
  s21: 0.140207
  s20: 0.231199



=== FD004: max per-unit std, sorted (lowest = most constant) ===
  s18: 0.000000
  s19: 0.000000
  s5: 0.000000
  s1: 0.000000
  s6: 0.817774
  s21: 1.015653
  s15: 1.059890
  s20: 1.073304
  s4: 1.083828
  s2: 1.103174



**Finding:** the constant-sensor set genuinely differs per dataset, not just by a little:

- **FD001:** 6 sensors at exactly 0.0 (s1, s5, s10, s16, s18, s19), then s6 at 0.0029, then a 15x jump to real signal at 0.043. Matches the documented 7-sensor list once s6 is included.
- **FD002 / FD004:** only 4 exact zeros (s1, s5, s18, s19). s6, s10, s16 are NOT constant here — their variance jumps straight to the same range as clearly-informative sensors. These sensors' readings are apparently tied to flight condition, which never changes in FD001/FD003 but varies constantly in FD002/FD004. This is exactly the case `dataset-reference.md` warns about: FD001's constant list does not transfer.
- **FD003:** 5 exact zeros, then s10 at 0.0055, then s6 at 0.037 (7x jump) — 6 constant sensors, one fewer than FD001. FD003 has a second fault mode (fan degradation) that FD001 doesn't, plausibly why s6 carries a little more signal here.

**Decision:** threshold of **0.01** on max per-unit std. It sits correctly in every dataset's gap — including FD003, the tightest case, where it needs to land between 0.0055 (constant) and 0.037 (real signal) — and clears the much larger gaps in the other three datasets with room to spare.